In [22]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect, brentq
from copy import copy

# Problem 1

In [23]:
class UpAndOutPut:

    def __init__(self, K, T, barrier, observationinterval):
        self.K = K
        self.T = T
        self.barrier = barrier
        self.observationinterval = observationinterval

In [24]:
hw2contract = UpAndOutPut(K=95, T=0.25, barrier=114, observationinterval=0.02)

In [25]:
class GBMdynamics:

    def __init__(self, S, r, rGrow, sigma=None):
        self.S = S
        self.X = S
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [26]:
# Use the same GBMdynamics class from HW 1

hw2dynamics = GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0)

In [27]:
class TreeEngine:

    def __init__(self, N):
        self.N = N

    def price_upandout(self, dynamics, contract):

        deltat = contract.T / self.N
        J = np.ceil(np.log(contract.barrier/dynamics.S)/(dynamics.sigma*np.sqrt(3*deltat))-0.5)
        deltax = np.log(contract.barrier/dynamics.S)/(J+0.5)

        Sgrid = dynamics.S*np.exp(np.linspace(self.N, -self.N, num=2*self.N+1, endpoint=True)*deltax)
        #Here I decided to make the SMALLER indexes in this array correspond to HIGHER S

        numTimestepsPerObs = contract.observationinterval/deltat
        if abs(numTimestepsPerObs-round(numTimestepsPerObs)) > 1e-8:
            raise ValueError("This value of N fails to place the observation dates in the tree.")

        # complete this
        nu = dynamics.rGrow - 0.5 * dynamics.sigma**2       
        Pu = 0.5 * ((dynamics.sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) + nu * deltat / deltax)       # complete this
        Pd = 0.5 * ((dynamics.sigma**2 * deltat + nu**2 * deltat**2) / (deltax**2) - nu * deltat / deltax)       # complete this
        Pm = 1 - Pu - Pd # shortcut 

        optionprice = np.maximum(contract.K-Sgrid,0)   #an array of time-T option prices.
        # print(optionprice)

        #Next, induct backwards to time 0, updating the optionprice array
        #Hint: if x is an array, then what are x[2:] and x[1:-1] and x[:-2]

        for t in np.linspace(self.N-1, 0, num=self.N, endpoint=True)*deltat:
            
            # insert lines of code here if needed

            # print(t)

            Sgrid = Sgrid[1:-1]

            optionprice = np.exp(-dynamics.r*deltat)*(Pu*optionprice[0:-2] + Pm*optionprice[1:-1] + Pd*optionprice[2:])

            if abs(t % contract.observationinterval) < 1e-8 and t > 0:
                optionprice = np.where(Sgrid >= contract.barrier, 0, optionprice)

        return optionprice[0]
        #The [0] is assuming that we are shrinking the optionprice array in each iteration of the loop,
        #until finally there is only 1 element in the array.
        #If instead you are keeping unchanged the size of the optionprice array in each iteration,
        #then you need to change the [0] to a different index.


In [28]:
hw2tree=TreeEngine(N=100)

hw2tree.price_upandout(hw2dynamics, hw2contract)

np.float64(5.350527169431585)

---
# 1 b

In [29]:
# Up-and-in put = Vanilla put - Up-and-out put
upandout_price = hw2tree.price_upandout(hw2dynamics, hw2contract)

bs_call = AnalyticEngine().BSpriceCall(
    GBMdynamics(S=100, sigma=0.4, rGrow=0, r=0),
    CallOption(K=95, T=0.25)
)
vanilla_put = bs_call - (100 - 95)

upandin_price = vanilla_put - upandout_price
upandin_price

np.float64(0.16901389424539026)

By **in-out parity**, the up-and-in and up-and-out barrier options together cover all scenarios, so their prices sum to the vanilla option price and therefore:

$$V_{\text{up-and-in}} = V_{\text{vanilla put}} - V_{\text{up-and-out}}$$

Using the Black-Scholes call price with $S_0 = 100$, $K = 95$, $T = 0.25$, $\sigma = 0.4$, $r = 0$:

$$V_{\text{vanilla put}} = V_{\text{call}}^{BS} - (100 - 95) = V_{\text{call}}^{BS} - 5$$

Substituting the numerical results:

$$V_{\text{up-and-in}} = V_{\text{vanilla put}} - V_{\text{up-and-out}} \approx 0.169$$

---
# 1 c

### 1c1

The continuously-monitored barrier option price is smaller. The continuous barrier can knock out at any time t in [0, 0.25], whereas the discrete barrier only checks at times $0.02, 0.04, \ldots, 0.24$. Since the continuous version has more opportunities to knock out, it knocks out more often, so its expected payoff and therefore price is lower.

### 1c2

The continuously-monitored up-and-out put can be replicated by:

- Long 1 vanilla put struck at $K = 95$
- Short $\alpha$ vanilla calls struck at $K^* = 136.8$

both with expiry $T = 0.25$.

When $S$ hits the barrier $H = 114$, the portfolio must have zero value:

$$\text{Put}(S=114, K=95, T-t) - \alpha \cdot \text{Call}(S=114, K^*=136.8, T-t) = 0$$

Using Black-Scholes with $r = 0$, the call and put pricing formulas give us a relationship that must hold for all remaining times $T - t$. Using the BS formula, at $S = H$:

$$\text{Put}(H, K) = K \cdot N(-d_2) - H \cdot N(-d_1)$$
$$\text{Call}(H, K^*) = H \cdot N(d_1^*) - K^* \cdot N(d_2^*)$$

Here $\log(H/K^*) = -\log(H/K)$. This symmetry of log-moneyness means:

$$d_1^* = -d_2, \quad d_2^* = -d_1$$

Therefore $N(d_1^*) = N(-d_2) = 1 - N(d_2)$ and $N(d_2^*) = N(-d_1) = 1 - N(d_1)$, giving:

$$\text{Call}(H, K^*) = H \cdot N(-d_2) - K^* \cdot N(-d_1)$$

Setting the portfolio to zero at $S = H$:

$$K \cdot N(-d_2) - H \cdot N(-d_1) = \alpha \left[ H \cdot N(-d_2) - K^* \cdot N(-d_1) \right]$$

Since $K^* = H^2/K$:

$$\alpha = \frac{K}{H} = \frac{95}{114}$$

# Problem 2

In [30]:
# Uses the CallOption and AnalyticEngine classes from HW 1

class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-$0$ vega

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma, because this function solves for sigma.
        # Returns time-$0$ implied volatility

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)  #discount factor
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)
        # We "try" values of sigma until we find sigma that generates price C

        # First find lower and upper bounds
        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2
        # We have calculated "lo" and "hi" which bound the implied volatility from below and above.
        # In other words, the implied volatility is somewhere in the interval [lo,hi].
        # Then, to calculate the implied volatility within that interval,
        # for purposes of this homework, you may either (A) write your own bisection algorithm,
        # or (B) use scipy.optimize.bisect or (C) use scipy.optimize.brentq
        # You will need to provide lo and hi to those solvers.
        # There are other solvers that do not require you to bound the solution
        # from below and above (for instance, scipy.optimize.fsolve is a useful solver).
        # However, if you are able to bound the solution (of a single-variable problem),
        # then bisection or Brent will be more reliable.

        impliedVolatility = brentq(lambda sigma: self.BSpriceCall(dynamics_try.update_sigma(sigma), contract) - C, lo, hi)

        return impliedVolatility
    
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

In [ ]:
hw2analytic = AnalyticEngine()
hw2dyn = GBMdynamics(sigma=None, rGrow=0, S=100, r=0)

# 2a
print("2a: Implied volatilities")
sigma_05 = hw2analytic.IV(hw2dyn, CallOption(K=100, T=0.5, price=11.25))
sigma_10 = hw2analytic.IV(hw2dyn, CallOption(K=100, T=1.0, price=12.00))
print("0.5-year IV:")
print(sigma_05)
print("1.0-year IV:")
print(sigma_10)

# 2b
print()
print("2b: 0.75-year price using interpolated vol")
sigma_075 = (sigma_05 + sigma_10) / 2
price_075 = hw2analytic.BSpriceCall(GBMdynamics(S=100, sigma=sigma_075, rGrow=0, r=0), CallOption(K=100, T=0.75))
print(sigma_075)
print(price_075)

print()
print("after linear interp of vol")
totalvar_075 = 0.5 * (sigma_05**2 * 0.5 + sigma_10**2 * 1.0)  # linear interp
sigma_075_correct = np.sqrt(totalvar_075 / 0.75)
price_075_correct = hw2analytic.BSpriceCall(GBMdynamics(S=100, sigma=sigma_075_correct, rGrow=0, r=0), CallOption(K=100, T=0.75))
print(sigma_075_correct)
print(price_075_correct)

2a: Implied volatilities
0.5-year IV:
0.4001327809210663
1.0-year IV:
0.3019384309935543

2b: 0.75-year price using interpolated vol
0.3510356059573103
12.081533286249247

after linear interp of vol
0.3378559232322385
11.631220250657627


# 2a

The implied volatilities with $S_0 = 100$, $K = 100$ (ATM), $r = 0$:

- 0.5-year call $\sigma_{0.5} \approx 0.4001$
- 1.0-year call $\sigma_{1.0} \approx 0.3019$

# 2b

We price the 0.75-year ATM call by linearly interpolating the implied volatilities:

$$\sigma_{0.75} = \frac{\sigma_{0.5} + \sigma_{1.0}}{2} = \frac{0.4001 + 0.3019}{2} \approx 0.3510$$

Plugging into Black-Scholes with $T = 0.75$:

$$V_{0.75}^{\text{interp}} = \text{BSCall}(S_0=100,\; K=100,\; T=0.75,\; \sigma=0.3510) \approx 12.08$$

# 2c

The price from (b) of $\approx 12.08$ is greater than the 1.0-year call price of 12.00. Since a longer-expiry ATM call must be worth at least as much as a shorter-expiry ATM call (with $r = 0$, the payoff $(S_T - K)^+$ is a convex function and more time can only add value), this violates no-arbitrage.

We arb with:

1. Sell the 0.75-year call at the (b) price of $12.08
2. Buy the 1.0-year call at $12.00
3. Pocket $12.08 - 12.00 = \$0.08$ upfront

At $T = 0.75$, if the 0.75-year call expires in the money with payoff $S_{0.75} - 100$, we owe that amount. But the 1.0-year call we hold is worth *at least* $(S_{0.75} - 100)^+$ at that time (it has 0.25 years remaining and is worth at least its intrinsic value). So the 1.0-year call covers our obligation, and we keep the $\$0.08$ risk-free profit.

Linearly interpolation gives total implied variance $\sigma^2 T$, not $\sigma$ itself:

$$\sigma_{0.75}^{\text{correct}} = \sqrt{0.0857 / 0.75} \approx 0.3379$$

This gives the no-arbitrage price:

$$V_{0.75}^{\text{correct}} = \text{BSCall}(S_0=100,\; K=100,\; T=0.75,\; \sigma=0.3379) \approx 11.63$$